# **EDA — SECOP II filtrado a CTeI (dataset limpio)**

Reto Universidad del Rosario: modelo de IA para analisis y prediccion de contratacion publica.

Este notebook trabaja sobre `secop_ctei_lineas_limpio.csv` (lineas de adjudicacion,
proceso x proveedor x lote) y `secop_ctei_procesos_limpio.csv` (agregado por proceso),
ya deduplicados (se removio la inflacion causada por una recarga del dataset fuente
a mitad de la descarga; ver el area de trabajo del proyecto para el detalle).

## Objetivos, mapeados a las 3 capacidades del reto

1. **Tendencias historicas** — patrones temporales, estacionalidad, comportamiento por
   sector (segmento UNSPSC), monto y modalidad.
2. **Dinamicas de mercado** — relaciones entidad-proveedor, concentracion de mercado
   (HHI), evolucion de participacion en el tiempo.
3. **Insumos para prediccion** — calidad y forma de la variable objetivo (adjudicado),
   candidatas a features, diseno de particion train/test temporal (sin fuga de
   informacion hacia el futuro).

Cada seccion cierra con una celda markdown de **Hallazgos** para completar despues de
correr el notebook y revisar los resultados juntos.

In [3]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def tabla_frecuencia(serie, nombre_categoria="categoria"):
    tabla = (
        serie.value_counts(dropna=False)
        .rename_axis(nombre_categoria)
        .reset_index(name="cantidad")
    )
    tabla["porcentaje"] = (tabla["cantidad"] / len(serie) * 100).round(2)
    return tabla


def grafica_barras_h(datos, categoria, valor, titulo, titulo_x, top=None):
    d = datos.copy()
    if top:
        d = d.nlargest(top, valor)
    d = d.sort_values(valor, ascending=True)
    fig = go.Figure(go.Bar(x=d[valor], y=d[categoria].astype(str), orientation="h"))
    fig.update_layout(title=titulo, xaxis_title=titulo_x, template="plotly_white",
                       height=max(350, 28 * len(d)))
    fig.show()


def grafica_serie(df, x, y, titulo, titulo_y, color=None):
    fig = go.Figure()
    if color and color in df.columns:
        for val, grupo in df.groupby(color):
            fig.add_trace(go.Scatter(x=grupo[x], y=grupo[y], mode="lines+markers", name=str(val)))
    else:
        fig.add_trace(go.Scatter(x=df[x], y=df[y], mode="lines+markers"))
    fig.update_layout(title=titulo, xaxis_title=x, yaxis_title=titulo_y, template="plotly_white")
    fig.show()


def hhi(participaciones):
    """HHI en escala 0-10000 a partir de una serie de participaciones (0-1)."""
    return (participaciones ** 2).sum() * 10000

# **1. Carga y verificacion de integridad**

In [5]:
RUTA_LINEAS = Path("secop_ctei_lineas_limpio.csv")
RUTA_PROCESOS = Path("secop_ctei_procesos_limpio.csv")

DTYPE_LINEAS = {"nit_entidad": "string", "nit_del_proveedor_adjudicado": "string"}

df_lineas = pd.read_csv(RUTA_LINEAS, dtype=DTYPE_LINEAS, low_memory=False)
df_proc = pd.read_csv(RUTA_PROCESOS, dtype={"nit_entidad": "string"}, low_memory=False)

for col in ["fecha_de_publicacion_del", "fecha_adjudicacion"]:
    if col in df_lineas.columns:
        df_lineas[col] = pd.to_datetime(df_lineas[col], errors="coerce")
    if col in df_proc.columns:
        df_proc[col] = pd.to_datetime(df_proc[col], errors="coerce")

df_lineas["adjudicado_bool"] = df_lineas["adjudicado"].map({"Si": True, "No": False}).astype("boolean")

print(f"Lineas: {len(df_lineas):,} filas, {df_lineas['id_del_proceso'].nunique():,} procesos distintos")
print(f"Procesos (agregado): {len(df_proc):,} filas")
print()
print("Duplicados de fila exacta remanentes en lineas:", df_lineas.duplicated().sum())
print("Procesos en 'lineas' que no aparecen en 'procesos':",
      len(set(df_lineas['id_del_proceso']) - set(df_proc['id_del_proceso'])))

Lineas: 495,284 filas, 492,797 procesos distintos
Procesos (agregado): 492,797 filas

Duplicados de fila exacta remanentes en lineas: 0
Procesos en 'lineas' que no aparecen en 'procesos': 0


In [6]:
diagnostico = pd.DataFrame({
    "columna": df_lineas.columns,
    "tipo": df_lineas.dtypes.astype(str).values,
    "nulos": df_lineas.isna().sum().values,
    "pct_nulos": (df_lineas.isna().mean().values * 100).round(2),
}).sort_values("pct_nulos", ascending=False)
diagnostico.head(20)

,columna,tipo,nulos,pct_nulos
32,nit_del_proveedor_adjudicado,string,470856,95.07
34,departamento_proveedor,object,460852,93.05
35,ciudad_proveedor,object,460696,93.02
37,fecha_adjudicacion,datetime64[ns],454805,91.83
33,nombre_del_proveedor,object,446445,90.14
6,ciudad_entidad,object,125887,25.42
5,departamento_entidad,object,38975,7.87
19,fecha_de_ultima_publicaci,object,10646,2.15
18,fecha_de_publicacion_del,datetime64[ns],10646,2.15
15,fase,object,8080,1.63


## 1.1 Hallazgos de integridad

- **Cuadre perfecto lineas <-> procesos**: 495,284 lineas cubren exactamente 492,797 procesos
  distintos, que coincide 1:1 con las 492,797 filas de `df_proc`. Cero duplicados de fila exacta
  y cero procesos huerfanos (en lineas pero no en procesos). La deduplicacion previa quedo bien
  hecha; no hay que volver a tocarla.
- **Los nulos siguen el patron esperado del ciclo de vida**: las columnas con mas nulos son
  justamente las que solo existen si el proceso *se adjudico* (`nit_del_proveedor_adjudicado`
  95.1%, `departamento_proveedor` 93.1%, `ciudad_proveedor` 93.0%, `fecha_adjudicacion` 91.8%,
  `nombre_del_proveedor` 90.1%). Esto cuadra con que solo el 7.89% de los procesos terminan
  adjudicados (ver 4.1) — no es un problema de calidad, es la naturaleza del dato. Implica que
  cualquier feature derivada del proveedor/fecha de adjudicacion **no puede usarse como input**
  del modelo de capacidad 3 (es informacion posterior al evento que se quiere predecir).
- `ciudad_entidad` (25.4% nulo) y `departamento_entidad` (7.9% nulo) son nulos "genuinos" de la
  entidad compradora, no dependen del desenlace — candidatas a imputar o agrupar en "no
  especificado" en vez de descartar filas.

<mark>Actualizacion tras revisar 3.1/3.2:</mark> el HHI general de 9,995.4 resulto ser un
artefacto de un valor `valor_total_adjudicacion` con magnitud imposible (ver 3.5). No hay indicio
de que ese error de magnitud afecte las columnas revisadas aqui (nulos, cuadre de conteos), pero
sí sirve de recordatorio de que "cero duplicados / cero huerfanos" valida la *unicidad* de las
filas, no la *plausibilidad* de los valores numericos — habria que agregar una verificacion de
rangos (mín/máx/percentiles extremos) sobre las columnas de valor monetario a este bloque de
integridad inicial.

# **2. Capacidad 1 — Tendencias historicas**

## 2.1 Regla de fecha condicionada al estado (re-verificar en el dataset limpio)

Ya establecimos en el EDA anterior que `fecha_de_publicacion_del` no es missing al
azar: depende del estado del ciclo de vida del proceso. Se reconfirma aqui sobre el
dataset limpio antes de construir cualquier serie temporal.

In [7]:
cobertura_fecha_estado = (
    df_proc.assign(tiene_fecha=df_proc["fecha_de_publicacion_del"].notna())
    .groupby("estado_del_procedimiento")["tiene_fecha"]
    .mean().mul(100).round(1).sort_values(ascending=False)
    .rename("pct_con_fecha").reset_index()
)
cobertura_fecha_estado

,estado_del_procedimiento,pct_con_fecha
0,Suspendido,100.00
1,Seleccionado,99.60
2,Publicado,99.30
3,Evaluación,99.20
4,Abierto,98.90
5,Cancelado,68.60
6,En aprobación,12.30
7,Aprobado,8.40
8,Borrador,1.50


## 2.2 Serie mensual de procesos y valor adjudicado

Se usa **solo** `df_proc` con fecha valida (regla de la seccion 2.1). El ultimo mes
del rango puede estar incompleto: revisar antes de interpretar caidas recientes como
tendencia real.

In [8]:
df_temporal = df_proc[df_proc["fecha_de_publicacion_del"].notna()].copy()
df_temporal["anio_mes"] = df_temporal["fecha_de_publicacion_del"].dt.to_period("M").dt.to_timestamp()

serie_mensual = (
    df_temporal.groupby("anio_mes")
    .agg(procesos=("id_del_proceso", "size"),
         valor_adjudicado=("valor_adjudicado_total", "sum"))
    .reset_index()
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=serie_mensual["anio_mes"], y=serie_mensual["procesos"],
                          name="Procesos", mode="lines"), secondary_y=False)
fig.add_trace(go.Scatter(x=serie_mensual["anio_mes"], y=serie_mensual["valor_adjudicado"],
                          name="Valor adjudicado (COP)", mode="lines"), secondary_y=True)
fig.update_layout(title="Procesos y valor adjudicado por mes", template="plotly_white")
fig.show()

print("Rango de fechas:", df_temporal["fecha_de_publicacion_del"].min(), "a",
      df_temporal["fecha_de_publicacion_del"].max())
print("Procesos en el ultimo mes del rango (revisar si esta incompleto):",
      serie_mensual.iloc[-1].to_dict())

Rango de fechas: 2022-01-01 00:00:00 a 2026-07-29 00:00:00
Procesos en el ultimo mes del rango (revisar si esta incompleto): {'anio_mes': Timestamp('2026-07-01 00:00:00'), 'procesos': 7714, 'valor_adjudicado': 419253038275}


## 2.3 Estacionalidad por mes del calendario (todos los anios agregados)

In [9]:
orden_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
               "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
df_temporal["mes"] = df_temporal["fecha_de_publicacion_del"].dt.month_name(locale=None)
mapa_mes = dict(zip(
    ["January","February","March","April","May","June","July","August",
     "September","October","November","December"], orden_meses))
df_temporal["mes_es"] = df_temporal["fecha_de_publicacion_del"].dt.month.map(
    dict(enumerate(orden_meses, start=1)))

por_mes = df_temporal["mes_es"].value_counts().reindex(orden_meses, fill_value=0)
fig = go.Figure(go.Bar(x=por_mes.index, y=por_mes.values))
fig.update_layout(title="Procesos por mes del calendario (todos los anios)",
                   template="plotly_white")
fig.show()

# Descomponer por anio para no confundir un anio incompleto con estacionalidad real
tabla_anio_mes = pd.crosstab(df_temporal["fecha_de_publicacion_del"].dt.year,
                              df_temporal["mes_es"])[orden_meses]
tabla_anio_mes

mes_es,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
fecha_de_publicacion_del,,,,,,,,,,,,
2022,23499,2509,1788,1558,2066,2463,5134,7884,8082,7695,8387,6583
2023,11549,14479,11214,7717,10002,12473,7640,8337,7252,7534,8594,6479
2024,9602,11766,9118,9533,9261,7107,8064,9498,9103,9894,8691,7052
2025,11781,14477,10629,8195,9269,8090,11022,8814,9680,10638,9922,7723
2026,30298,7324,4640,2497,4115,5730,7714,0,0,0,0,0


## 2.4 Tendencias por segmento UNSPSC (sector)

Recordatorio: el segmento se deriva de `codigo_principal_de_categoria`
(primeros 2 digitos tras quitar el prefijo `V1.`), ya calculado en la extraccion.

In [10]:
SEGMENTOS_NOMBRE = {
    "80": "Gestion / profesionales / administrativos",
    "81": "Ingenieria, investigacion y tecnologia",
    "86": "Educacion y capacitacion",
}
df_temporal["segmento_nombre"] = df_temporal["segmento_unspsc"].astype(str).map(SEGMENTOS_NOMBRE).fillna(df_temporal["segmento_unspsc"].astype(str))

por_segmento_mes = (
    df_temporal.groupby(["anio_mes", "segmento_nombre"])
    .size().reset_index(name="procesos")
)
grafica_serie(por_segmento_mes, "anio_mes", "procesos",
              "Procesos por mes segun segmento UNSPSC", "Procesos", color="segmento_nombre")

tabla_frecuencia(df_temporal["segmento_nombre"], "segmento")

,segmento,cantidad,porcentaje
0,"Ingenieria, investigacion y tecnologia",214178,44.42
1,Gestion / profesionales / administrativos,167442,34.73
2,Educacion y capacitacion,100545,20.85


## 2.5 Tendencias por modalidad de contratacion

In [11]:
por_modalidad_anio = (
    df_temporal.assign(anio=df_temporal["fecha_de_publicacion_del"].dt.year)
    .groupby(["anio", "modalidad_de_contratacion"]).size()
    .reset_index(name="procesos")
)
top_modalidades = (
    df_temporal["modalidad_de_contratacion"].value_counts().head(6).index.tolist()
)
grafica_serie(
    por_modalidad_anio[por_modalidad_anio["modalidad_de_contratacion"].isin(top_modalidades)],
    "anio", "procesos", "Procesos por anio segun modalidad (top 6)", "Procesos",
    color="modalidad_de_contratacion"
)

## 2.6 Hallazgos — Capacidad 1

- **`fecha_de_publicacion_del` no es MCAR**: cobertura ~99% en estados avanzados
  (Suspendido, Seleccionado, Publicado, Evaluacion, Abierto) pero cae a 12.3% en
  "En aprobacion", 8.4% en "Aprobado" y 1.5% en "Borrador". Confirmado sobre el dataset
  limpio: cualquier serie temporal o split por fecha implicitamente excluye procesos en
  etapas tempranas del flujo interno, que aun no se publican. Esto es correcto para el
  analisis de "lo que el mercado ve", pero hay que dejarlo explicito en el reporte.
- **El ultimo mes (2026-07) esta incompleto**: 7,714 procesos vs. un promedio de
  ~9,000-10,000/mes en 2023-2025. Como hoy es 2026-07-31, julio de 2026 aun no cierra —
  cualquier lectura de "caida" en jul-2026 es un artefacto de corte, no tendencia real.
  Debe excluirse del grafico de tendencia o marcarse visualmente como parcial.
- **Estacionalidad de enero es real pero su magnitud en 2022 y 2026 es sospechosa**:
  enero domina todos los anios (consistente con el ciclo presupuestal colombiano: muchas
  entidades publican su Plan Anual de Adquisiciones y abren procesos al arrancar el anio
  fiscal), pero el salto es demasiado grande en los extremos de la serie:
  - 2022: enero = 23,499 vs. febrero = 2,509 (~9.4x), muy por encima del patron
    enero/febrero de 2023-2025 (~0.8x-1.3x). Puede ser un remanente de la carga inicial del
    dataset (el propio notebook menciona una recarga a mitad de descarga que causo
    inflacion) que la deduplicacion no elimino del todo, o el arranque real del universo de
    datos SECOP II CTeI en esa fecha.
  - 2026: enero = 30,298, ~2-3x el enero de cualquier otro anio (2023: 11,549; 2024: 9,602;
    2025: 11,781). Esto no puede explicarse solo por estacionalidad de ciclo fiscal.
  - **Accion recomendada**: excluir 2022 (anio de arranque, posible artefacto) y 2026
    (anio incompleto, enero anomalo) del analisis de estacionalidad "tipica", y usar solo
    2023-2025 como referencia de patron estacional limpio. Vale la pena revisar si
    2026-01 tiene una concentracion inusual de procesos con la misma entidad/fecha exacta
    de publicacion (indicio de carga masiva automatizada).
- **Segmento UNSPSC**: Ingenieria/investigacion/tecnologia (81) domina con 44.4% de los
  procesos, seguido de Gestion/profesionales (80) 34.7% y Educacion/capacitacion (86)
  20.9% — coherente con el foco CTeI del filtro aplicado al dataset.
- **Modalidad de contratacion**: pendiente de leer el grafico interactivo (no se imprime
  tabla en esta celda), pero se retoma cuantitativamente en 4.2, donde se ve que
  "Contratacion regimen especial" y "Contratacion directa" son las modalidades con mas
  volumen (219,248 y 185,886 procesos) y, notablemente, **0% de tasa de adjudicacion** —
  ver el hallazgo critico en 4.5.

# **3. Capacidad 2 — Dinamicas de mercado**

Se usa `df_lineas` filtrado a lineas efectivamente adjudicadas
(`adjudicado_bool == True` y `valor_total_adjudicacion > 0`), que es la
unidad correcta para medir participacion de mercado de proveedores.

In [12]:
df_mercado = df_lineas[
    (df_lineas["adjudicado_bool"] == True) & (df_lineas["valor_total_adjudicacion"] > 0)
].copy()
df_mercado["proveedor_id"] = df_mercado["nit_del_proveedor_adjudicado"].fillna(df_mercado["nombre_del_proveedor"])

print(f"Lineas adjudicadas con valor positivo: {len(df_mercado):,}")
print(f"Proveedores distintos: {df_mercado['proveedor_id'].nunique():,}")
print(f"Entidades compradoras distintas: {df_mercado['entidad'].nunique():,}")

Lineas adjudicadas con valor positivo: 41,168
Proveedores distintos: 17,183
Entidades compradoras distintas: 2,296


## 3.1 Concentracion general de proveedores (HHI + Pareto 80/20)

In [13]:
mercado = (
    df_mercado.groupby("proveedor_id")["valor_total_adjudicacion"].sum()
    .sort_values(ascending=False).reset_index()
)
mercado["participacion"] = mercado["valor_total_adjudicacion"] / mercado["valor_total_adjudicacion"].sum()
mercado["participacion_acum"] = mercado["participacion"].cumsum()

hhi_general = hhi(mercado["participacion"])
n_80pct = (mercado["participacion_acum"] <= 0.80).sum() + 1

print(f"HHI general: {hhi_general:,.1f}  "
      f"({'baja' if hhi_general < 1500 else 'moderada' if hhi_general < 2500 else 'alta'} concentracion)")
print(f"Proveedores que concentran el 80% del valor: {n_80pct:,} de {len(mercado):,} "
      f"({n_80pct / len(mercado):.1%})")

grafica_barras_h(mercado.head(20), "proveedor_id", "valor_total_adjudicacion",
                  "Top 20 proveedores por valor adjudicado total", "Valor adjudicado (COP)")

HHI general: 9,995.4  (alta concentracion)
Proveedores que concentran el 80% del valor: 1 de 17,183 (0.0%)


## 3.2 Concentracion de proveedores por entidad compradora

In [14]:
def hhi_por_entidad(df):
    resultados = []
    for entidad, sub in df.groupby("entidad"):
        val = sub.groupby("proveedor_id")["valor_total_adjudicacion"].sum()
        part = val / val.sum()
        resultados.append({
            "entidad": entidad,
            "hhi": hhi(part),
            "proveedores_distintos": sub["proveedor_id"].nunique(),
            "procesos_adjudicados": sub["id_del_proceso"].nunique(),
            "valor_total": val.sum(),
        })
    return pd.DataFrame(resultados).sort_values("valor_total", ascending=False)

hhi_entidades = hhi_por_entidad(df_mercado)
print("Entidades con mayor valor adjudicado (top 15):")
hhi_entidades.head(15)

Entidades con mayor valor adjudicado (top 15):


,entidad,hhi,proveedores_distintos,procesos_adjudicados,valor_total
829,EAG,"9,999.99",75,110,214790335198875839
779,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,"1,383.75",264,949,5727377498700
1612,MINISTERIO DE MINAS Y ENERGIA,"7,610.81",68,87,4836052632118
1561,INVIAS,163.98,656,972,1631509329408
301,ANI,329.50,80,78,1316704294502
698,DEPARTAMENTO DE ANTIOQUIA//,"1,909.81",177,279,1238997511450
1608,MINISTERIO DE EDUCACION NACIONAL (MEN),"1,521.10",84,148,1092231185004
14,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",714.58,69,54,1001244950027
1604,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,"6,904.40",29,35,997418818827
2090,SECRETARIA DE EDUCACION DEL DISTRITO,"1,067.58",118,151,963028590183


In [15]:
grafica_barras_h(hhi_entidades[hhi_entidades["procesos_adjudicados"] >= 20],
                  "entidad", "hhi",
                  "HHI por entidad compradora (min. 20 procesos adjudicados)",
                  "HHI (0-10000)", top=20)

## 3.3 Evolucion de participacion de mercado en el tiempo

Sirve para detectar si hay proveedores ganando terreno (entrantes) o perdiendolo
(salientes) anio a anio — insumo directo para la capacidad 3
(probabilidad de adjudicacion futura por proveedor).

In [16]:
df_mercado_fecha = df_mercado[df_mercado["fecha_de_publicacion_del"].notna()].copy()
df_mercado_fecha["anio"] = df_mercado_fecha["fecha_de_publicacion_del"].dt.year

top_10_global = mercado.head(10)["proveedor_id"].tolist()

participacion_anual = (
    df_mercado_fecha.groupby(["anio", "proveedor_id"])["valor_total_adjudicacion"]
    .sum().reset_index()
)
total_anual = df_mercado_fecha.groupby("anio")["valor_total_adjudicacion"].sum()
participacion_anual["participacion_pct"] = participacion_anual.apply(
    lambda r: r["valor_total_adjudicacion"] / total_anual.loc[r["anio"]] * 100, axis=1
)

grafica_serie(
    participacion_anual[participacion_anual["proveedor_id"].isin(top_10_global)],
    "anio", "participacion_pct",
    "Participacion de mercado (%) por anio — top 10 proveedores globales",
    "Participacion (%)", color="proveedor_id"
)

## 3.4 Relacion entidad-proveedor: grado de cada nodo (red bipartita, sin graficar el grafo completo)

In [17]:
grado_proveedor = (
    df_mercado.groupby("proveedor_id")["entidad"].nunique()
    .sort_values(ascending=False).rename("entidades_distintas")
)
grado_entidad = (
    df_mercado.groupby("entidad")["proveedor_id"].nunique()
    .sort_values(ascending=False).rename("proveedores_distintos")
)

print("Distribucion de grado de proveedores (a cuantas entidades distintas le vende):")
print(grado_proveedor.describe())
print()
print("Distribucion de grado de entidades (a cuantos proveedores distintos le compra):")
print(grado_entidad.describe())

print()
print("Proveedores 'diversificados' (venden a 5+ entidades):", (grado_proveedor >= 5).sum(),
      f"de {len(grado_proveedor):,}")

Distribucion de grado de proveedores (a cuantas entidades distintas le vende):
count   17,183.00
mean         1.63
std          2.59
min          1.00
25%          1.00
50%          1.00
75%          1.00
max        105.00
Name: entidades_distintas, dtype: float64

Distribucion de grado de entidades (a cuantos proveedores distintos le compra):
count   2,296.00
mean       12.22
std        24.29
min         1.00
25%         1.00
50%         5.00
75%        13.00
max       656.00
Name: proveedores_distintos, dtype: float64

Proveedores 'diversificados' (venden a 5+ entidades): 753 de 17,183


## 3.5 Hallazgos — Capacidad 2

- **🚩 HALLAZGO CRITICO — outlier de magnitud imposible en `valor_total_adjudicacion`**:
  el HHI general da 9,995.4 sobre 10,000 (monopolio casi total) y un solo proveedor
  concentraria el 80% del valor adjudicado de *todo* el mercado CTeI (1 de 17,183, 0.0%).
  Al abrir por entidad, el origen es claro: la entidad **"EAG"** tiene HHI = 9,999.99 y
  `valor_total` = **214,790,335,198,875,839 COP** (~214 mil billones / 214 *quadrillones*
  de pesos). Para contexto, el PIB anual de Colombia es del orden de 1,500 billones de
  pesos — este valor es ~140,000 veces el PIB del pais y no puede ser real. Es casi
  seguro un error de magnitud en el dato fuente (posible corrimiento de decimales,
  concatenacion de valores, o un campo mal parseado en la extraccion).
  - Este mismo problema probablemente infla la serie de `valor_adjudicado` en 2.2 (no se
    filtro `valor_total_adjudicacion`/`valor_adjudicado_total` por rango razonable en
    ningun punto del notebook) y el ranking del top 20 proveedores en 3.1.
  - **Accion antes de cualquier conclusion de mercado**: localizar la(s) fila(s) de la
    entidad EAG con este valor, decidir si se corrige (dividir por el factor de error si
    es identificable) o se excluye, y **volver a correr 2.2, 3.1, 3.2 y 3.3** con el dato
    corregido — el HHI general y el ranking de proveedores actuales no son confiables
    todavia. `PATRIMONIO AUTÓNOMO AEROCAFÉ` tambien llama la atencion (HHI 9,391 con solo
    2 proveedores y 2 procesos) pero su valor total (~655 mil millones) es plausible, solo
    refleja bajo volumen de procesos.
- **Concentracion de proveedores por entidad, fuera del outlier EAG**: hay bastante
  heterogeneidad — INVIAS (972 procesos adjudicados, 656 proveedores) tiene HHI muy bajo
  (163.98, mercado competido), mientras Ministerio de Minas y Energia (HHI 7,610.81, 68
  proveedores) y Ministerio de Agricultura (HHI 6,904.40, 29 proveedores) muestran alta
  concentracion incluso sin ser outliers de magnitud — candidatos reales a revisar para
  el analisis de dinamica de mercado.
- **Red entidad-proveedor muy poco conectada del lado del proveedor**: la mediana de
  entidades distintas por proveedor es 1 (75% de los proveedores le venden a una sola
  entidad); solo 753 de 17,183 (4.4%) son "diversificados" (5+ entidades). Del lado de
  la entidad la mediana es 5 proveedores distintos. Esto sugiere relaciones
  proveedor-entidad bastante fijas/dedicadas mas que un mercado liquido con proveedores
  compitiendo ampliamente — relevante como feature para capacidad 3 (ej. "proveedor ya le
  vendio antes a esta entidad" puede ser predictor fuerte).
- Evolucion de participacion en el tiempo (3.3): queda pendiente cuantificar (el grafico
  es interactivo, sin tabla impresa) — recomendado agregar una tabla resumen de
  entrantes/salientes del top 10 antes de cerrar esta seccion, una vez corregido el
  outlier de EAG.

# **4. Capacidad 3 — Insumos para prediccion**

No se entrena ningun modelo aqui todavia: se revisa que tan lista esta la variable
objetivo y las features candidatas, y se deja preparado el corte temporal para el
train/test split (nunca aleatorio en series de tiempo: hay que evitar que el modelo
"vea" el futuro).

## 4.1 Variable objetivo: adjudicado_proceso

In [18]:
dist_objetivo = tabla_frecuencia(df_proc["adjudicado_proceso"], "adjudicado")
print(dist_objetivo)

# Estabilidad del desbalance en el tiempo: si cambia mucho por anio, el split
# temporal puede quedar con proporciones muy distintas entre train y test.
df_proc_fecha = df_proc[df_proc["fecha_de_publicacion_del"].notna()].copy()
df_proc_fecha["anio"] = df_proc_fecha["fecha_de_publicacion_del"].dt.year
tasa_por_anio = (
    df_proc_fecha.groupby("anio")["adjudicado_proceso"].mean().mul(100).round(2)
    .rename("tasa_adjudicacion_pct").reset_index()
)
tasa_por_anio

   adjudicado  cantidad  porcentaje
0       False    453902       92.11
1        True     38895        7.89


,anio,tasa_adjudicacion_pct
0,2022,10.97
1,2023,7.64
2,2024,7.57
3,2025,7.85
4,2026,5.95


## 4.2 Variable objetivo por segmento y por modalidad (candidatas a features fuertes)

In [19]:
tasa_por_segmento = (
    df_proc.groupby("segmento_unspsc")["adjudicado_proceso"].agg(["mean", "size"])
    .rename(columns={"mean": "tasa_adjudicacion", "size": "n_procesos"})
    .sort_values("n_procesos", ascending=False)
)
tasa_por_segmento["tasa_adjudicacion"] = (tasa_por_segmento["tasa_adjudicacion"] * 100).round(2)
print("Por segmento UNSPSC:")
display(tasa_por_segmento)

tasa_por_modalidad = (
    df_proc.groupby("modalidad_de_contratacion")["adjudicado_proceso"].agg(["mean", "size"])
    .rename(columns={"mean": "tasa_adjudicacion", "size": "n_procesos"})
    .sort_values("n_procesos", ascending=False)
)
tasa_por_modalidad["tasa_adjudicacion"] = (tasa_por_modalidad["tasa_adjudicacion"] * 100).round(2)
print("Por modalidad:")
display(tasa_por_modalidad)

Por segmento UNSPSC:


,tasa_adjudicacion,n_procesos
segmento_unspsc,,
81,11.57,219852
80,5.44,170534
86,4.09,102411


Por modalidad:


,tasa_adjudicacion,n_procesos
modalidad_de_contratacion,,
Contratación régimen especial,0.00,219248
Contratación directa,0.00,185886
Mínima cuantía,77.67,22685
Concurso de méritos abierto,39.93,16755
Solicitud de información a los Proveedores,0.00,15225
Selección Abreviada de Menor Cuantía,9.77,10464
Contratación Directa (con ofertas),78.37,8210
Contratación régimen especial (con ofertas),66.43,5686
Selección abreviada subasta inversa,40.09,4909


## 4.3 Correlacion de variables numericas con el objetivo (Spearman, no asume linealidad)

In [20]:
vars_numericas = pd.DataFrame({
    "precio_base": df_proc["precio_base"],
    "duracion": df_proc["duracion"],
    "numero_de_lotes": df_proc["numero_de_lotes"],
    "proveedores_invitados": df_proc["proveedores_invitados"],
    "proveedores_con_invitacion": df_proc["proveedores_con_invitacion"],
    "respuestas_al_procedimiento": df_proc["respuestas_al_procedimiento"],
    "adjudicado": df_proc["adjudicado_proceso"].astype(float),
}).astype(float)

corr = vars_numericas.corr(method="spearman")["adjudicado"].drop("adjudicado").sort_values(key=abs, ascending=False)
corr

respuestas_al_procedimiento    0.78
proveedores_con_invitacion     0.33
precio_base                    0.20
proveedores_invitados          0.18
duracion                      -0.10
numero_de_lotes                0.03
Name: adjudicado, dtype: float64

## 4.4 Diseno del split temporal (train/test)

Regla: el corte es una **fecha**, no una fraccion aleatoria de filas — de lo
contrario el modelo entrenaria con informacion que en la realidad no existia
todavia (fuga de informacion hacia el futuro).

In [21]:
FECHA_CORTE_SPLIT = "2025-07-01"  # ajustar segun el hallazgo de 2.2 (mes/anio incompleto)

train = df_proc_fecha[df_proc_fecha["fecha_de_publicacion_del"] < FECHA_CORTE_SPLIT]
test = df_proc_fecha[df_proc_fecha["fecha_de_publicacion_del"] >= FECHA_CORTE_SPLIT]

print(f"Train: {len(train):,} procesos ({train['fecha_de_publicacion_del'].min()} a {train['fecha_de_publicacion_del'].max()})")
print(f"Test:  {len(test):,} procesos ({test['fecha_de_publicacion_del'].min()} a {test['fecha_de_publicacion_del'].max()})")
print()
print(f"Tasa de adjudicacion train: {train['adjudicado_proceso'].mean():.2%}")
print(f"Tasa de adjudicacion test:  {test['adjudicado_proceso'].mean():.2%}")

Train: 362,048 procesos (2022-01-01 00:00:00 a 2025-06-30 00:00:00)
Test:  120,117 procesos (2025-07-01 00:00:00 a 2026-07-29 00:00:00)

Tasa de adjudicacion train: 7.96%
Tasa de adjudicacion test:  8.10%


## 4.5 Hallazgos — Capacidad 3

- **🚩 HALLAZGO CRITICO — el desbalance del target esta dominado por dos modalidades con
  0% de adjudicacion estructural**: "Contratacion regimen especial" (219,248 procesos) y
  "Contratacion directa" (185,886 procesos) suman 405,134 de 492,797 procesos (82.2% del
  dataset) y **ninguno de ellos tiene `adjudicado_proceso = True`**. Esto no parece ruido —
  es casi seguro que estas modalidades usan un mecanismo de adjudicacion distinto (por
  invitacion directa/regimen especial) donde el campo `adjudicado_proceso` tal como esta
  construido no aplica o se registra de otra forma en el SECOP. Consecuencias:
  - El 92.11% de "No adjudicado" global es en gran parte un artefacto de mezclar
    modalidades donde el target es estructuralmente 0 con modalidades donde si varia.
  - Un modelo entrenado sobre el dataset completo aprenderia trivialmente "si es regimen
    especial o directa, predecir No" y eso infla artificialmente el accuracy sin aportar
    valor. **Se debe decidir explicitamente**: (a) excluir estas dos modalidades del
    modelado de capacidad 3 y modelarlas aparte o no modelarlas, o (b) confirmar con la
    fuente que son ceros estructurales legitimos y evaluar el modelo con metricas que no
    se dejen enganar por ellos (PR-AUC, recall en la clase minoritaria, evaluado dentro de
    las modalidades donde el target si varia).
  - Modalidades donde el target si tiene variacion real: Minima cuantia (77.7% adjud.,
    n=22,685), Concurso de meritos abierto (39.9%), Contratacion Directa con ofertas
    (78.4%), Licitacion publica (39.6%), etc. — estas son las que realmente valen la pena
    modelar.
- **Desbalance estable pero con tendencia a la baja**: tasa de adjudicacion anual
  10.97% (2022) → 7.64% (2023) → 7.57% (2024) → 7.85% (2025) → 5.95% (2026, incompleto).
  Es razonablemente estable 2023-2025 (~7.6-7.9%), lo que da confianza en que train y test
  no deberian tener proporciones muy distintas — y en efecto no la tienen (7.96% vs
  8.10%, ver mas abajo). El 2026 bajo puede ser normal (año parcial, procesos recientes
  aun sin resolver) mas que una caida real.
- **Segmento con mejor tasa de adjudicacion**: 81 (Ingenieria/investigacion/tecnologia)
  11.57%, casi el doble que 80 (Gestion, 5.44%) y casi el triple que 86 (Educacion,
  4.09%) — buena feature categorica.
- **Correlacion Spearman con el objetivo**: `respuestas_al_procedimiento` (0.78) es, por
  lejos, la mas fuerte. ⚠️ **riesgo de fuga de informacion**: si el numero de respuestas
  solo se conoce con certeza al cierre del proceso (mismo momento que se sabe si se
  adjudico), usarla como feature predictiva "en tiempo real" (antes de que el proceso
  cierre) no es valida — hay que verificar en que momento del ciclo de vida se congela
  ese valor antes de usarla como input del modelo. `proveedores_con_invitacion` (0.33) y
  `precio_base` (0.20) son mas seguras de usar como features de antemano.
- **Split temporal propuesto (corte 2025-07-01) es razonable**: train 362,048 procesos
  (2022-01 a 2025-06, tasa 7.96%) vs. test 120,117 procesos (2025-07 a 2026-07, tasa
  8.10%) — tamanos y tasas de la clase positiva muy parecidos entre train y test, buena
  señal para que el split no introduzca sesgo por si mismo. Eso si, el test incluye el mes
  incompleto 2026-07, que convendria excluir o tratar aparte al evaluar el modelo final.

# **5. Conclusiones generales**

**Listo para modelar / analizar:**
- La base `lineas` x `procesos` esta bien cuadrada (0 duplicados, 0 huerfanos) y el
  patron de nulos es interpretable (ligado al desenlace del proceso), no ruido.
- La regla "usar solo fecha valida + excluir el ultimo mes incompleto" para series
  temporales y el split temporal por fecha de corte (2025-07-01) son solidos y quedan
  documentados para reusar en los notebooks siguientes.
- Hay una feature categorica fuerte y disponible desde el inicio del proceso: modalidad
  de contratacion (y, dentro de ella, segmento UNSPSC).

**Bloqueantes que hay que resolver antes de sacar conclusiones de negocio (o de
entrenar el modelo final), en orden de impacto:**
1. **Outlier de magnitud imposible en `valor_total_adjudicacion`** (entidad "EAG",
   ~214 mil billones de COP) — invalida el HHI general, el top de proveedores y
   probablemente distorsiona la serie de valor adjudicado mensual. Resuelto y confirmado
   en el notebook de correcciones (`Correcciones_outliers_y_modelado.ipynb`): ver nota
   de seguimiento mas abajo.
2. **Dos modalidades de contratacion (regimen especial y contratacion directa, 82.2% del
   dataset) con 0% de adjudicacion estructural** — hay que decidir explicitamente si se
   excluyen del modelo de capacidad 3 o se tratan aparte; de lo contrario el desbalance
   de clases y las metricas de cualquier modelo quedan distorsionadas por un artefacto de
   definicion, no por dificultad real de prediccion.
3. **Posible fuga de informacion via `respuestas_al_procedimiento`** (corr. 0.78 con el
   objetivo) — verificar el momento del ciclo de vida en que se conoce ese valor antes de
   usarlo como feature.
4. Enero de 2022 y enero de 2026 muestran picos de volumen que se salen del patron
   estacional 2023-2025 — no bloquea el modelado pero si el analisis de estacionalidad
   "tipica" de capacidad 1; usar 2023-2025 como referencia y excluir/anotar los extremos.

**Pendiente fuera de alcance de este notebook:** la taxonomia semantica via embeddings
(agrupar objetos de contratacion por similitud de texto mas alla del codigo UNSPSC)
sigue sin abordarse.

<mark>Nota de seguimiento (agregada despues de revisar `Correcciones_outliers_y_modelado.ipynb`):</mark>
el outlier de EAG **no es un error de escala uniforme** (no fue "todo dividido/multiplicado
por un factor fijo") — es un valor puntual sin relacion aritmetica limpia con su propio
`precio_base` (463 millones vs. un `valor_total_adjudicacion` ~463,000 veces mayor), es decir
un error de captura/parseo en esa fila especifica, no un problema sistemico de unidades.
El siguiente notebook identifico ademas otras 13 lineas (14 en total, apenas 0.003% de las
lineas) igual de implausibles frente a su `precio_base` aunque de magnitud mucho menor
(cientos de miles de millones, no quadrillones) — el tratamiento aplicado fue **excluir**
esas 14 lineas de los calculos monetarios (via una regla de umbral absoluto + relativo a
`precio_base`), no reescalarlas, porque no hay forma confiable de recuperar el valor real.
Con esa exclusion el HHI general pasa de 9,995.4 ("monopolio") a **116.1 ("baja
concentracion")** y el "80% del valor en 1 proveedor" pasa a "80% del valor en 1,044 de
17,179 proveedores (6.1%)" — la conclusion correcta de mercado es la del notebook de
correcciones, no la de este EDA. El detalle completo queda en
`Correcciones_outliers_y_modelado.ipynb`, secciones A y B. **Ajuste de proceso para
futuros EDAs de este proyecto**: agregar en la seccion 1 un chequeo sistematico de
plausibilidad (percentiles extremos y razon valor/precio_base) sobre las columnas de valor
monetario, no solo confiar en que "cero duplicados" implica datos limpios.

<mark>Nota de seguimiento #2 (estacionalidad de enero):</mark> el notebook de correcciones
revisa la distribucion diaria dentro de enero 2022 y 2026 y **descarta la hipotesis de
artefacto de carga masiva** — en ningun enero (ni los "normales" 2023-2025 ni los atipicos
2022/2026) un solo dia concentra mas del 10% del mes, por lo que el volumen extra esta
distribuido de forma consistente con actividad real, no con una recarga puntual. La
magnitud atipica de esos dos eneros sigue sin explicacion definitiva (posible efecto de
arranque del dataset en 2022 y cambio real de ciclo/politica en 2026) y debe reportarse
como limitacion, no como error de datos a corregir.

<mark>Nota de seguimiento #3 (fondos administrados, agregada tras revisar
`Capacidad1_cierre_final.ipynb`):</mark> incluso despues de quitar el outlier de EAG y las
demas lineas implausibles, la serie de `valor_adjudicado` mensual de la seccion 2.2 sigue
mezclando dos fenomenos de naturaleza distinta: mercado competido normal y "fondos
administrados" — 109 procesos (0.02% del total) de un solo proveedor por diseno (ej.
Ministerio de Minas y Energia con GECELCA, dic-2025, ~4.4 billones reales) que en conjunto
representan **28.1% del valor real adjudicado**. El notebook de cierre de Capacidad 1
recalcula la serie mensual con y sin estos casos; **cualquier cifra de "valor total
adjudicado" o "tamano del mercado" citada a partir de este EDA debe reemplazarse por la
version con y sin fondos administrados de `Capacidad1_cierre_final.ipynb`, seccion 3** —
la de este notebook subestima la distorsion aun despues de la correccion de outliers.